# Autoresearch Experiment Analysis

Analysis of autonomous GLOBEM depression prediction experiments from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, balanced_acc, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["balanced_acc"] = pd.to_numeric(df["balanced_acc"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    acc = row["balanced_acc"]
    desc = row["description"]
    print(f"  #{i:3d}  bal_acc={acc:.4f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Balanced Accuracy Over Time

Track how the best (kept) balanced_acc evolves as experiments progress. The running maximum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_acc = valid.loc[0, "balanced_acc"]

# Only plot points near or above baseline (the interesting region)
above = valid[valid["balanced_acc"] >= baseline_acc - 0.005]

# Plot discarded as faint background dots
disc = above[above["status"] == "DISCARD"]
ax.scatter(disc.index, disc["balanced_acc"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = above[above["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["balanced_acc"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_acc = valid.loc[kept_mask, "balanced_acc"]
running_max = kept_acc.cummax()
best = running_max.iloc[-1] if len(running_max) > 0 else baseline_acc
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, acc in zip(kept_idx, kept_acc):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, acc),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Balanced Accuracy (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below baseline to just above best
margin = (best - baseline_acc) * 0.15 if best > baseline_acc else 0.01
ax.set_ylim(baseline_acc - margin, best + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_acc = df.iloc[0]["balanced_acc"]
best_acc = kept["balanced_acc"].max()
best_row = kept.loc[kept["balanced_acc"].idxmax()]

print(f"Baseline balanced_acc:  {baseline_acc:.4f}")
print(f"Best balanced_acc:      {best_acc:.4f}")
print(f"Total improvement:      {best_acc - baseline_acc:.4f} ({(best_acc - baseline_acc) / baseline_acc * 100:.2f}%)")
print(f"Best experiment:        {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: bal_acc={row['balanced_acc']:.4f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's balanced_acc
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_acc"] = kept["balanced_acc"].shift(1)
kept["delta"] = kept["balanced_acc"] - kept["prev_acc"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Bal Acc':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.4f}  {row['balanced_acc']:.4f}      {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.4f}  {'':>10}  TOTAL improvement over baseline")